# Ingestion Using Auto Loader

## Import and Define Paths and Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

schema = StructType([
    StructField("Invoice", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("Price", DecimalType(12, 4), True),
    StructField("Customer ID", DoubleType(), True),
    StructField("Country", StringType(), True)
])

source_path = "/Volumes/retail_project/landing/raw_files/"
target_path = "retail_project.bronze.transactions"


## Read the files with autoloader

In [0]:
df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(schema)
    .load(source_path)
)

df_bronze = df_bronze.withColumnRenamed(
    "Customer ID",
    "CustomerID"
)

## Add ingestion metadata

In [0]:
df_bronze = (
    df_bronze
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp())
)

## Write to delta table

In [0]:
checkpoint_path = "/Volumes/retail_project/landing/pipeline_state/transactions_checkpoint"

query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("retail_project.bronze.transactions")
)

query.awaitTermination()